In [ ]:
# 1) stringoutputParser
# 2) jsonoutparser
# 3) StructuredOutputParser
# 4) PydanticOutputParser

In [38]:
import os
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated , List , Optional
from datetime import datetime
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser , JsonOutputParser 
from langchain.output_parsers import StructuredOutputParser , ResponseSchema , PydanticOutputParser

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash" , api_key= GOOGLE_API_KEY)
llm_gemini.invoke("who is father of india")

AIMessage(content='Mahatma Gandhi is widely considered the Father of India.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--b8ea4363-7aab-4af9-88c3-872a4f7ba355-0', usage_metadata={'input_tokens': 5, 'output_tokens': 13, 'total_tokens': 18, 'input_token_details': {'cache_read': 0}})

In [ ]:
# woking with model.content instead of stroutputParser
template1 = PromptTemplate(
    template= "write a deatailed report on the {topic}",
    input_variables= ['topic']
    )

template2 = PromptTemplate(
    template= "Give me 5 point summary for this given text: \n {text}",
    input_variables= ['text']
    )

promt1 = template1.invoke({'topic' : "cricket"})
result1 = llm_gemini.invoke(promt1)

promt2 = template2.invoke({'text' : result1.content})
final_result = llm_gemini.invoke(promt2)

print(final_result.content)

In [23]:
# woking stroutputParser clean syntax and much eaiser
template1 = PromptTemplate(
    template= "write a deatailed report on the {topic}",
    input_variables= ['topic']
    )

template2 = PromptTemplate(
    template= "Give me 5 point summary for this given text: \n {text}",
    input_variables= ['text']
    )

parser = StrOutputParser()

chain = template1 | llm_gemini | parser | template2 | llm_gemini | parser

print(chain.invoke("chess"))

Here's a 5-point summary of the provided text:

1.  **Chess is an ancient strategy game:** Originating in 6th-century India as Chaturanga, chess evolved through Persia and Europe to become the game we know today, with standardized rules and international governance by FIDE.
2.  **The game is played with specific rules and pieces:** Chess involves two players aiming to checkmate the opponent's king on an 8x8 board, utilizing pieces with unique movement capabilities and special moves like castling and en passant.
3.  **Chess requires strategic and tactical thinking:** Successful play involves understanding opening principles, middlegame strategies, endgame techniques, and tactical motifs like forks, pins, and sacrifices.
4.  **Chess notation allows for game recording and analysis:** Algebraic notation provides a standardized system for documenting moves, facilitating study and review of chess games.
5.  **Chess is culturally significant:** Chess permeates literature, art, and education, 

In [ ]:
# JsonOutputParser --> problem is we can not inforce out own schema 
json_parser = JsonOutputParser()
template = PromptTemplate(
    template= "give me the name, age and city of any friction\n {inst}",
    input_variables= [],
    partial_variables= {'inst' : json_parser.get_format_instructions()}
)

promt = template.format()
# result = llm_gemini.invoke(promt)
# print(json_parser.parse(result.content))

chain = template | llm_gemini | json_parser
print(chain.invoke({}))

{'name': 'Alex Johnson', 'age': 28, 'city': 'Springfield'}


In [ ]:
# StructedOutputParser --> disadvantage ( No data validation is not possible )
schema = [
    ResponseSchema(name= "fact_1", description= "fact 1 about the topic"),
    ResponseSchema(name= "fact_2", description= "fact 2 about the topic"),
    ResponseSchema(name= "fact_3", description= "fact 3 about the topic") 
]

structured_parser = StructuredOutputParser.from_response_schemas(schema)

promt3 = PromptTemplate(
    template= 'Give me 3 amzing facts about {topic} \n {format_instr}',
    input_variables= ['topic'],
    partial_variables= {"format_instr" : structured_parser.get_format_instructions()}
)

chain = promt3 | llm_gemini | structured_parser

print(chain.invoke("backhole"))

{'fact_1': "Black holes aren't cosmic vacuum cleaners! While they have intense gravity, you'd only get sucked in if you got too close. At a safe distance, you could orbit a black hole just like planets orbit a star.", 'fact_2': "Time slows down near a black hole. This is due to the extreme gravity warping spacetime. An observer watching something fall into a black hole would see it appear to slow down and eventually freeze at the event horizon, although from the falling object's perspective, time would pass normally.", 'fact_3': 'The smallest black holes are thought to have formed in the early universe and could be as small as an atom but with the mass of a mountain! These are called primordial black holes and their existence is still hypothetical.'}


In [44]:
# PydanticOutputParser
from pydantic import BaseModel , Field

class Person(BaseModel):
    name : str = Field(description= 'name of the person')
    age : int = Field(gt=18 , description= "age of the person should be more than 18")
    city : str = Field(description= "name of the person city")
    
pydantic_parser = PydanticOutputParser(pydantic_object= Person)

template4 = PromptTemplate(
    template= "give me the name, age and city of any friction\n {inst} based of the county of {country}",
    input_variables= ['country'],
    partial_variables= {'inst' : pydantic_parser.get_format_instructions()}
)


chain = template4 | llm_gemini | pydantic_parser

print(chain.invoke("US"))

name='Jane Doe' age=35 city='Anytown'
